In [1]:
import torch
from torch import nn

In [2]:

import sentencepiece as spm

text = "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence"
embedding = nn.Embedding(32000, 100)
# input_embedding(text)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input_ids = tokenizer.encode(text, return_tensors="pt")

d:\anaconda3\envs\deeplearning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\anaconda3\envs\deeplearning\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
input_embedding = embedding(input_ids)

In [4]:
pos = torch.arange(0,31)
# pos = (pos.unsqueeze(1).repeat(1, 100))
# pos.shape
pos


tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])

In [5]:
i = torch.arange(0,50)
d = torch.pow(10000, 2*i/100)
pos = torch.arange(0,31)
pos = (pos.unsqueeze(1).repeat(1, 50))

In [ ]:
arg = (pos / d)
sin_val = torch.sin(arg)
cos_val = torch.cos(arg)


In [7]:
stacked = torch.stack((sin_val, cos_val), dim=2)
stacked
final_pe = stacked.flatten(1)
final_pe

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  7.3912e-01,  ...,  1.0000e+00,
          1.2023e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.9570e-01,  ...,  1.0000e+00,
          2.4045e-04,  1.0000e+00],
        ...,
        [ 2.7091e-01, -9.6261e-01, -9.6308e-01,  ...,  9.9999e-01,
          3.3663e-03,  9.9999e-01],
        [-6.6363e-01, -7.4806e-01, -8.4768e-01,  ...,  9.9999e-01,
          3.4866e-03,  9.9999e-01],
        [-9.8803e-01,  1.5425e-01, -1.7886e-01,  ...,  9.9999e-01,
          3.6068e-03,  9.9999e-01]])

In [8]:
x = input_embedding + final_pe 

In [11]:
wq = torch.rand(100, 100)
q = x @ wq 
q.shape

wk = torch.rand(100, 100)
k = x @ wk 
k.shape

wv = torch.rand(100, 100)
v = x @ wv 
v.shape

# ins = (q @ k.transpose(-2, -1))/100
# softmax = nn.Softmax(dim=-1)
# att = softmax(ins) @ v
# att.shape

torch.Size([1, 31, 100])

In [ ]:
# wq = torch.rand(31, 100)
# q = torch.rand(31, 100)
# k = torch.rand(31, 100)
# v = torch.rand(31, 100)
w = {}
for i in range(8):
    w[f'wiq {i}'] = torch.rand(100, 100)
    w[f'wik {i}'] = torch.rand(100, 100)
    w[f'wiv {i}'] = torch.rand(100, 100)

multihead = None
for i in range(8):
    qm = q @ w[f'wiq {i}']

    km = k @ w[f'wik {i}']

    vm = v @ w[f'wiv {i}']

    ins = (qm @ km.transpose(-2, -1))/100
    softmax = nn.Softmax(dim=-1)
    att = softmax(ins) @ vm
    if multihead == None:
        multihead = att
    else:
        print(multihead.shape)
        multihead = torch.cat((multihead, att), 2)
    print(att.shape)

torch.Size([1, 31, 100])
torch.Size([1, 31, 100])
torch.Size([1, 31, 100])
torch.Size([1, 31, 200])
torch.Size([1, 31, 100])
torch.Size([1, 31, 300])
torch.Size([1, 31, 100])
torch.Size([1, 31, 400])
torch.Size([1, 31, 100])
torch.Size([1, 31, 500])
torch.Size([1, 31, 100])
torch.Size([1, 31, 600])
torch.Size([1, 31, 100])
torch.Size([1, 31, 700])
torch.Size([1, 31, 100])


In [28]:
w[f'wo'] = torch.rand(8*100, 100)
multihead_final = multihead @ w[f'wo']

In [26]:
multihead.shape

torch.Size([1, 31, 800])

In [29]:
multihead_final.shape

torch.Size([1, 31, 100])

In [ ]:
class TransformerTranslation(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.w = nn.ParameterDict({})

        for i in range(6):
            for j in range(8):
                self.w[f'wiq_{i}_{j}'] = nn.Parameter(torch.randn(100, 100))
                self.w[f'wik_{i}_{j}'] = nn.Parameter(torch.randn(100, 100))
                self.w[f'wiv_{i}_{j}'] = nn.Parameter(torch.randn(100, 100))
                
            self.w[f'wq_{i}'] = nn.Parameter(torch.rand(100, 100))
            self.w[f'wk_{i}'] = nn.Parameter(torch.rand(100, 100))
            self.w[f'wv_{i}'] = nn.Parameter(torch.rand(100, 100))
            # h * dv, dm
            self.w[f'wo_{i}'] = nn.Parameter(torch.rand(8*100, 100))
            
        self.embedding = nn.Embedding(32000, 100)
        self.layernorm_1 = nn.LayerNorm(100)
        self.linear_1 = nn.Linear(100, 2048)
        self.relu_1 = nn.ReLU(2048)
        self.linear_2 = nn.Linear(2048, 100)
        self.layernorm_2 = nn.LayerNorm(100)
        # self.layernorm_1 = nn.Sequential(nn.Linear(1, 2), nn.Linear(2, 3))

    def forward(self, x):
        x = self.embedding(x)
        x_output = self.positional_embedding(x)

        for i in range(6):
            q = x_output @ self.w[f'wq_{i}']
            k = x_output @ self.w[f'wk_{i}']
            v = x_output @ self.w[f'wv_{i}']
            x = self.attention(q, k , v, i, self.w[f'wo_{i}'])
            print(f'x : {x.shape}')
            x_output = self.layernorm_1(x_output + x)
            print(f'x : {x.shape}')
            x = self.linear_1(x)
            print(f'x : {x.shape}')
            x = self.relu_1(x)
            print(f'x : {x.shape}')
            x = self.linear_2(x)
            print(f'x : {x.shape}')
            x = self.layernorm_2(x_output + x)
            print(f'x hasil akhir: {x.shape}')
            return x
    
    def positional_embedding(self, x):
        pos = torch.arange(0,31)
        i = torch.arange(0,50)
        d = torch.pow(10000, 2*i/100)
        pos = torch.arange(0,31)
        pos = (pos.unsqueeze(1).repeat(1, 50))
        arg = (pos / d)
        sin_val = torch.sin(arg)
        cos_val = torch.cos(arg)
        stacked = torch.stack((sin_val, cos_val), dim=2)
        stacked
        final_pe = stacked.flatten(1)
        final_pe
        x = input_embedding + final_pe 
        return x
    
    def attention(self, q, k, v, i, wo):
        multihead = None
        for j in range(8):
            qm = q @ self.w[f'wiq_{i}_{j}']

            km = k @ self.w[f'wik_{i}_{j}']

            vm = v @ self.w[f'wiv_{i}_{j}']

            ins = (qm @ km.transpose(-2, -1))/torch.sqrt(torch.tensor(100))
            softmax = nn.Softmax(dim=-1)
            att = softmax(ins) @ vm
            if multihead == None:
                multihead = att
            else:
                print(multihead.shape)
                multihead = torch.cat((multihead, att), 2)
            print(att.shape)
        multihead_final = multihead @ wo
        return multihead_final

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer

def positional_embedding(x):
        pos = torch.arange(0,31)
        i = torch.arange(0,50)
        d = torch.pow(10000, 2*i/100)
        pos = torch.arange(0,31)
        pos = (pos.unsqueeze(1).repeat(1, 50))
        arg = (pos / d)
        sin_val = torch.sin(arg)
        cos_val = torch.cos(arg)
        stacked = torch.stack((sin_val, cos_val), dim=2)
        stacked
        final_pe = stacked.flatten(1)
        final_pe
        x = input_embedding + final_pe 
        return x
    
class Attention(nn.Module):
    def __init__(self, dmodel, dk):
        super(Attention, self).__init__()
        # self.wiq = nn.Parameter(torch.randn(dmodel, dk))
        # self.wik = nn.Parameter(torch.randn(dmodel, dk))
        # self.wiv = nn.Parameter(torch.randn(dmodel, dk))
        self.w = nn.ParameterDict({})
        # h*dv aslinya tapi aku anggep dk dv sama untuk sekarang
        self.wo = nn.Parameter(torch.empty(8*dk, dmodel))
        nn.init.xavier_uniform_(self.wo)
        for i in range(8):
            self.w[f'wiq_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiq_{i}'])
            self.w[f'wik_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wik_{i}'])
            self.w[f'wiv_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiv_{i}'])
    
    def forward(self, q, k, v, mask=None):
        multihead = None
        for i in range(8):
            qm = q @ self.w[f'wiq_{i}'] 

            km = k @ self.w[f'wik_{i}'] 

            vm = v @ self.w[f'wiv_{i}'] 

            scores = (qm @ km.transpose(-2, -1))/ 10.0
            if mask != None:
                # print(f"scores shape : {scores.shape}")
                # print(f"Mask shape      : {mask.shape}")
                mask_expanded = mask.unsqueeze(1)  # shape [2, 1, 31] to broadcast over query dim
                scores = scores.masked_fill(mask_expanded == 0, float('-inf'))
            softmax = nn.Softmax(dim=-1)
            att = softmax(scores) @ vm
            if multihead == None:
                multihead = att
            else:
                # print(multihead.shape)
                multihead = torch.cat((multihead, att), 2)
            # print(att.shape)
        multihead_final = multihead @ self.wo
        return multihead_final

        
    
class Transformer(nn.Module):
    def __init__(self):
        super(Transformer, self).__init__()
        self.w = nn.ParameterDict({})

        self.wq = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wq)
        self.wk = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wk)
        self.wv = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wv)
            
        self.attention_1 = Attention(100, 100)
        self.layernorm_1 = nn.LayerNorm(100)
        self.linear_1 = nn.Linear(100, 2048)
        self.relu_1 = nn.ReLU()
        self.linear_2 = nn.Linear(2048, 100)
        self.layernorm_2 = nn.LayerNorm(100)
        
    def forward(self, x_output, mask=None):
        # x = self.embedding(x)
        # x_output = self.positional_embedding(x)
        # q = x_output @ self.wq
        # k = x_output @ self.wk
        # v = x_output @ self.wv
        if mask == None:
            x = self.attention_1(x_output, x_output, x_output)
        else:
            x = self.attention_1(x_output, x_output, x_output, mask)
        # print(f'x : {x.shape}')
        x_output = self.layernorm_1(x_output + x)
        # print(f'x : {x.shape}')
        x = self.linear_1(x_output)
        # print(f'x : {x.shape}')
        x = self.relu_1(x)
        # print(f'x : {x.shape}')
        x = self.linear_2(x)
        # print(f'x : {x.shape}')
        x = self.layernorm_2(x_output + x)
        # print(f'x hasil akhir: {x.shape}')
        return x
    
class TransformerTranslation(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(32000, 100)
        # self.pos_embedding = nn.Embedding(50, 100)
        self.layers_input = nn.ModuleList([Transformer() for _ in range(2)])
        self.linear = nn.Linear(100, 3)
        self.softmax = nn.Softmax()

    def forward(self, x):
        batch_size, seq_len = x.shape
        
        mask = torch.where((x == 0), x, 1)
        
        # Embeddings
        x_emb = self.embedding(x)
        
        # Position Embeddings
        # positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
        # x_pos = self.pos_embedding(positions)
        x_pos = positional_embedding(x)
        
        x = x_emb + x_pos
        
        # x = self.embedding(x)
        # x = self.positional_embedding(x)
        for layer in self.layers_input:
            x = layer(x, mask)
        # print(f'x shape sebelum linear : {x.shape}')
        x = torch.mean(x, dim=1)
        # print(f'x shape sesudah mean  : {x.shape}')
        x = self.linear(x)
        # x = self.softmax(x)
        return x

    # def positional_embedding(self, input_embedding):
    #     pos = torch.arange(0,31)
    #     i = torch.arange(0,50)
    #     d = torch.pow(10000, 2*i/100)
    #     pos = torch.arange(0,31)
    #     pos = (pos.unsqueeze(1).repeat(1, 50))
    #     arg = (pos / d)
    #     sin_val = torch.sin(arg)
    #     cos_val = torch.cos(arg)
    #     stacked = torch.stack((sin_val, cos_val), dim=2)
    #     stacked
    #     final_pe = stacked.flatten(1)
    #     final_pe
    #     x = input_embedding.to(device) + final_pe.to(device)
    #     return x
    

    

model = TransformerTranslation()

# text = "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence"
# text_indo = "Karena model kami tidak menggunakan rekursi maupun konvolusi, agar model dapat memanfaatkan urutan dari deretan (sequence)."

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# input_ids = tokenizer.encode(text, return_tensors="pt")
# y_pred = model(input_ids)
# target = torch.randn_like(y_pred)
# loss = F.mse_loss(y_pred, target)

# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()


In [212]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split

# ==========================================
# 1. PPKM DATASET CLASS (Reads your CSV)
# ==========================================
class PPKMDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.tokenizer = tokenizer
        self.data = df
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Use .iloc to access rows by integer index
        row = self.data.iloc[idx]
        text = str(row['Tweet'])      # Your column name
        label = int(row['sentiment']) # Your column name (0, 1, 2)
        
        # Tokenize
        enc = self.tokenizer.encode_plus(
            text, 
            max_length=31, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), torch.tensor(label)


# ==========================================
# 3. LOAD & SPLIT DATA
# ==========================================
# Read the file
filename = r"D:\download_d\INA_TweetsPPKM_Labeled_Pure.csv\INA_TweetsPPKM_Labeled_Pure.csv"
try:
    print(f"Loading {filename}...")
    full_df = pd.read_csv(filename, sep='\t')
    print(f"Total rows: {len(full_df)}")
    
    # Split 80% Train, 20% Test
    train_df, test_df = train_test_split(full_df, test_size=0.2, random_state=42)
    print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")
    
except Exception as e:
    print(f"Error loading file: {e}")
    # Fallback dummy data if file is missing
    train_df = pd.DataFrame({'Tweet': ['Dummy text'], 'sentiment': [1]})
    test_df = pd.DataFrame({'Tweet': ['Dummy text'], 'sentiment': [1]})

# Setup DataLoaders
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
train_dataset = PPKMDataset(train_df, tokenizer)
test_dataset = PPKMDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

model = TransformerTranslation().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # Low LR for stability
criterion = nn.CrossEntropyLoss()

epochs = 5
print("\nStarting Training on PPKM Tweets...")

for epoch in range(1, epochs + 1):
    # --- TRAIN ---
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in train_loader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    # --- TEST ---
    model.eval()
    test_correct = 0
    test_total = 0
    
    with torch.no_grad():
        for input_ids, labels in test_loader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            outputs = model(input_ids)
            preds = torch.argmax(outputs, dim=1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    train_acc = 100 * correct / total
    test_acc = 100 * test_correct / test_total
    
    print(f"Epoch {epoch}: Loss = {train_loss/len(train_loader):.4f} | Train Acc = {train_acc:.2f}% | Val Acc = {test_acc:.2f}%")

Loading D:\download_d\INA_TweetsPPKM_Labeled_Pure.csv\INA_TweetsPPKM_Labeled_Pure.csv...
Total rows: 23644
Train size: 18915 | Test size: 4729
Running on: cuda

Starting Training on PPKM Tweets...
Epoch 1: Loss = 0.6257 | Train Acc = 76.29% | Val Acc = 77.82%
Epoch 2: Loss = 0.5362 | Train Acc = 78.75% | Val Acc = 79.13%
Epoch 3: Loss = 0.4708 | Train Acc = 81.35% | Val Acc = 78.98%
Epoch 4: Loss = 0.4010 | Train Acc = 84.29% | Val Acc = 79.62%
Epoch 5: Loss = 0.3262 | Train Acc = 87.43% | Val Acc = 80.69%


In [97]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

# ==========================================
# 1. DATASET
# ==========================================
class SimpleDataset(Dataset):
    def __init__(self):
        # 0 = Negative, 1 = Neutral, 2 = Positive
        self.data = [
            ("I love this movie", 2), ("Great film", 2), ("Awesome", 2), 
            ("I hate this", 0), ("Terrible", 0), ("Bad movie", 0), 
            ("It was okay", 1), ("Not bad", 1), ("Average", 1)
        ] * 20 # Duplicate data to make training loop last longer
        self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        enc = self.tokenizer.encode_plus(
            text, max_length=31, padding='max_length', truncation=True, return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), torch.tensor(label)

# ==========================================
# 2. YOUR MODEL (CLEANED & FIXED)
# ==========================================
class Attention(nn.Module):
    def __init__(self, dmodel, dk):
        super(Attention, self).__init__()
        # self.wiq = nn.Parameter(torch.randn(dmodel, dk))
        # self.wik = nn.Parameter(torch.randn(dmodel, dk))
        # self.wiv = nn.Parameter(torch.randn(dmodel, dk))
        self.w = nn.ParameterDict({})
        # h*dv aslinya tapi aku anggep dk dv sama untuk sekarang
        self.wo = nn.Parameter(torch.empty(8*dk, dmodel))
        nn.init.xavier_uniform_(self.wo)
        for i in range(8):
            self.w[f'wiq_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiq_{i}'])
            self.w[f'wik_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wik_{i}'])
            self.w[f'wiv_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
            nn.init.xavier_uniform_(self.w[f'wiv_{i}'])
    
    def forward(self, q, k, v):
        multihead = None
        for i in range(8):
            qm = q @ self.w[f'wiq_{i}'] 

            km = k @ self.w[f'wik_{i}'] 

            vm = v @ self.w[f'wiv_{i}'] 

            ins = (qm @ km.transpose(-2, -1))/ 10.0
            softmax = nn.Softmax(dim=-1)
            att = softmax(ins) @ vm
            if multihead == None:
                multihead = att
            else:
                # print(multihead.shape)
                multihead = torch.cat((multihead, att), 2)
            # print(att.shape)
        multihead_final = multihead @ self.wo
        return multihead_final

 
class Transformer(nn.Module):
    def __init__(self):
        super(Transformer, self).__init__()
        self.w = nn.ParameterDict({})

        self.wq = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wq)
        self.wk = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wk)
        self.wv = nn.Parameter(torch.empty(100, 100))
        nn.init.xavier_uniform_(self.wv)
            
        self.attention_1 = Attention(100, 100)
        self.layernorm_1 = nn.LayerNorm(100)
        self.linear_1 = nn.Linear(100, 2048)
        self.relu_1 = nn.ReLU()
        self.linear_2 = nn.Linear(2048, 100)
        self.layernorm_2 = nn.LayerNorm(100)
        
    def forward(self, x_output):
        # x = self.embedding(x)
        # x_output = self.positional_embedding(x)
        # q = x_output @ self.wq
        # k = x_output @ self.wk
        # v = x_output @ self.wv
        x = self.attention_1(x_output, x_output, x_output)
        # print(f'x : {x.shape}')
        x_output = self.layernorm_1(x_output + x)
        # print(f'x : {x.shape}')
        x = self.linear_1(x_output)
        # print(f'x : {x.shape}')
        x = self.relu_1(x)
        # print(f'x : {x.shape}')
        x = self.linear_2(x)
        # print(f'x : {x.shape}')
        x = self.layernorm_2(x_output + x)
        # print(f'x hasil akhir: {x.shape}')
        return x
    

# class TransformerTranslation(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.embedding = nn.Embedding(32000, 100)
#         self.pos_embedding = nn.Embedding(50, 100)
#         self.layers_input = nn.ModuleList([Transformer() for _ in range(2)]) # 2 Layers is enough
#         self.linear = nn.Linear(100, 3)

#     def forward(self, x):
#         batch_size, seq_len = x.shape
        
#         # Embeddings
#         x_emb = self.embedding(x)
#         positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
#         x_pos = self.pos_embedding(positions)
#         x = x_emb + x_pos
        
#         # Layers
#         for layer in self.layers_input:
#             x = layer(x)
            
#         # Max Pooling (Best for classification)
#         x, _ = torch.max(x, dim=1)
        
#         # Output
#         x = self.linear(x)
#         return x
    
class TransformerTranslation(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(32000, 100)
        self.pos_embedding = nn.Embedding(50, 100)
        self.layers_input = nn.ModuleList([Transformer() for _ in range(2)])
        self.linear = nn.Linear(100, 3)
        self.softmax = nn.Softmax()

    def forward(self, x):
        batch_size, seq_len = x.shape
        
        # Embeddings
        x_emb = self.embedding(x)
        
        # Position Embeddings
        positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
        x_pos = self.pos_embedding(positions)
        
        x = x_emb + x_pos
        
        # x = self.embedding(x)
        # x = self.positional_embedding(x)
        for layer in self.layers_input:
            x = layer(x)
        # print(f'x shape sebelum linear : {x.shape}')
        x, _ = torch.max(x, dim=1)
        # print(f'x shape sesudah mean  : {x.shape}')
        x = self.linear(x)
        # x = self.softmax(x)
        return x
    
    # ufdtid68======================================================
    
# class Attention(nn.Module):
#     def __init__(self, dmodel, dk):
#         super(Attention, self).__init__()
#         # self.wiq = nn.Parameter(torch.randn(dmodel, dk))
#         # self.wik = nn.Parameter(torch.randn(dmodel, dk))
#         # self.wiv = nn.Parameter(torch.randn(dmodel, dk))
#         self.w = nn.ParameterDict({})
#         # h*dv aslinya tapi aku anggep dk dv sama untuk sekarang
#         self.wo = nn.Parameter(torch.empty(8*dk, dmodel))
#         nn.init.xavier_uniform_(self.wo)
#         for i in range(8):
#             self.w[f'wiq_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
#             nn.init.xavier_uniform_(self.w[f'wiq_{i}'])
#             self.w[f'wik_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
#             nn.init.xavier_uniform_(self.w[f'wik_{i}'])
#             self.w[f'wiv_{i}'] = nn.Parameter(torch.empty(dmodel, dk))
#             nn.init.xavier_uniform_(self.w[f'wiv_{i}'])
    
#     def forward(self, q, k, v):
#         multihead = None
#         for i in range(8):
#             qm = q @ self.w[f'wiq_{i}'] 

#             km = k @ self.w[f'wik_{i}'] 

#             vm = v @ self.w[f'wiv_{i}'] 

#             ins = (qm @ km.transpose(-2, -1))/ 10.0
#             softmax = nn.Softmax(dim=-1)
#             att = softmax(ins) @ vm
#             if multihead == None:
#                 multihead = att
#             else:
#                 # print(multihead.shape)
#                 multihead = torch.cat((multihead, att), 2)
#             # print(att.shape)
#         multihead_final = multihead @ self.wo
#         return multihead_final

        
    
# class Transformer(nn.Module):
#     def __init__(self):
#         super(Transformer, self).__init__()
#         self.w = nn.ParameterDict({})

#         self.wq = nn.Parameter(torch.empty(100, 100))
#         nn.init.xavier_uniform_(self.wq)
#         self.wk = nn.Parameter(torch.empty(100, 100))
#         nn.init.xavier_uniform_(self.wk)
#         self.wv = nn.Parameter(torch.empty(100, 100))
#         nn.init.xavier_uniform_(self.wv)
            
#         self.attention_1 = Attention(100, 100)
#         self.layernorm_1 = nn.LayerNorm(100)
#         self.linear_1 = nn.Linear(100, 2048)
#         self.relu_1 = nn.ReLU()
#         self.linear_2 = nn.Linear(2048, 100)
#         self.layernorm_2 = nn.LayerNorm(100)
        
#     def forward(self, x_output):
#         # x = self.embedding(x)
#         # x_output = self.positional_embedding(x)
#         # q = x_output @ self.wq
#         # k = x_output @ self.wk
#         # v = x_output @ self.wv
#         x = self.attention_1(x_output, x_output, x_output)
#         # print(f'x : {x.shape}')
#         x_output = self.layernorm_1(x_output + x)
#         # print(f'x : {x.shape}')
#         x = self.linear_1(x_output)
#         # print(f'x : {x.shape}')
#         x = self.relu_1(x)
#         # print(f'x : {x.shape}')
#         x = self.linear_2(x)
#         # print(f'x : {x.shape}')
#         x = self.layernorm_2(x_output + x)
#         # print(f'x hasil akhir: {x.shape}')
#         return x
    
# class TransformerTranslation(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.embedding = nn.Embedding(32000, 100)
#         self.pos_embedding = nn.Embedding(50, 100)
#         self.layers_input = nn.ModuleList([Transformer() for _ in range(6)])
#         self.linear = nn.Linear(100, 3)
#         self.softmax = nn.Softmax()

#     def forward(self, x):
#         batch_size, seq_len = x.shape
        
#         # Embeddings
#         x_emb = self.embedding(x)
        
#         # Position Embeddings
#         positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)
#         x_pos = self.pos_embedding(positions)
        
#         x = x_emb + x_pos
        
#         # x = self.embedding(x)
#         # x = self.positional_embedding(x)
#         for layer in self.layers_input:
#             x = layer(x)
#         # print(f'x shape sebelum linear : {x.shape}')
#         x, _ = torch.max(x, dim=1)
#         # print(f'x shape sesudah mean  : {x.shape}')
#         x = self.linear(x)
#         # x = self.softmax(x)
#         return x

# ==========================================
# 3. EXECUTION (This resets everything)
# ==========================================


# [Image of Transformer Encoder Architecture]


# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Re-Initialize Model & Data
model = TransformerTranslation().to(device)
dataset = SimpleDataset()
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# Re-Initialize Optimizer (Crucial Step!)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Starting fresh training...")
for epoch in range(1, 16):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in dataloader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch}: Loss = {total_loss/len(dataloader):.4f}, Accuracy = {acc:.2f}%")

Running on: cuda
Starting fresh training...
Epoch 1: Loss = 1.4034, Accuracy = 33.89%
Epoch 2: Loss = 1.1631, Accuracy = 28.33%
Epoch 3: Loss = 0.9150, Accuracy = 53.89%
Epoch 4: Loss = 0.4829, Accuracy = 81.67%
Epoch 5: Loss = 0.0449, Accuracy = 100.00%
Epoch 6: Loss = 0.0041, Accuracy = 100.00%
Epoch 7: Loss = 0.0029, Accuracy = 100.00%
Epoch 8: Loss = 0.0024, Accuracy = 100.00%
Epoch 9: Loss = 0.0021, Accuracy = 100.00%
Epoch 10: Loss = 0.0019, Accuracy = 100.00%
Epoch 11: Loss = 0.0017, Accuracy = 100.00%
Epoch 12: Loss = 0.0015, Accuracy = 100.00%
Epoch 13: Loss = 0.0014, Accuracy = 100.00%
Epoch 14: Loss = 0.0013, Accuracy = 100.00%
Epoch 15: Loss = 0.0012, Accuracy = 100.00%


In [197]:
from torch.utils.data import Dataset, DataLoader
import torch

class ToySentimentDataset(Dataset):
    def __init__(self, tokenizer):
        # 0 = Negative, 1 = Neutral, 2 = Positive
        self.tokenizer = tokenizer
        
        # YOUR 50 SENTENCES (from before)
        self.sentences = [
            "I love this movie so much", "What a horrible film", "It was okay, not the best", 
            "Absolutely fantastic experience", "Terrible plot and bad acting", "Mediocre and boring",
            "I enjoyed every moment", "I hate every part of it", "Not good, not bad",
            "It was great and fun", "Awful and disappointing", "Satisfying but could be better",
            "Loved the characters and story", "Worst movie ever", "Quite average film",
            "Amazing visuals, great soundtrack", "Bad direction ruined it", "Neutral feelings about this",
            "I really liked it", "I don't dislike it", "It is not my type of movie",
            "Wonderful and heartwarming", "Terrible ending", "Nothing special",
            "Fantastic pace and acting", "Poorly written script", "I feel indifferent",
            "Wonderful, I recommend it", "Not worth watching", "Decent enough",
            "Excellent movie overall", "It lacked depth", "Good but not perfect",
            "Too many flaws", "Pretty fun to watch", "Would not watch again",
            "I’m on the fence", "Brilliant performance", "Disappointing experience",
            "Balanced — some good, some bad", "Loved some parts, hated others",
            "Fine for a relaxed evening", "Terrible from start to finish", "Nothing memorable",
            "Awesome cinematic journey", "Mediocre acting", "Not bad at all",
            "You should watch it", "I don’t recommend this movie", "Neutral review"
        ]

        self.labels = [
            2,0,1,2,0,1,2,0,1,2,
            0,1,2,0,1,2,0,1,2,1,
            1,2,0,1,2,0,1,2,0,1,
            2,0,1,2,0,2,0,1,2,0,
            1,0,1,2,0,1,2,1,2,1
        ]

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        # We handle tokenization here to keep the loop clean
        enc = self.tokenizer.encode_plus(
            self.sentences[idx],
            max_length=31,            # Fixed length like your manual code expected
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return enc['input_ids'].squeeze(0), torch.tensor(self.labels[idx])

In [198]:
# 1. Setup Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Initialize the dataset using the class above
# CHANGE: Use ToySentimentDataset instead of SimpleDataset
dataset = ToySentimentDataset(tokenizer) 
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# 3. Setup Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerTranslation().to(device) # Make sure class is defined previously

# 4. Setup Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f"Training on {len(dataset)} sentences...")

# 5. Run Training
for epoch in range(1, 31): # 30 Epochs to be safe
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for input_ids, labels in dataloader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}: Loss = {total_loss/len(dataloader):.4f}, Accuracy = {acc:.2f}%")

Training on 50 sentences...
Epoch 1: Loss = 1.6339, Accuracy = 38.00%
Epoch 5: Loss = 1.1209, Accuracy = 30.00%
Epoch 10: Loss = 0.4073, Accuracy = 76.00%
Epoch 15: Loss = 0.2911, Accuracy = 86.00%
Epoch 20: Loss = 0.0462, Accuracy = 100.00%
Epoch 25: Loss = 0.0115, Accuracy = 100.00%
Epoch 30: Loss = 0.0064, Accuracy = 100.00%
